In [1]:
%pip install pandasql

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd

#Modify the path according to where the file is in your filesystem
dataset_path = "data-3.csv" 

df = pd.read_csv(dataset_path)

In [3]:
print("Shape:", df.shape)
print("Data types:")
print(df.dtypes)
print("Missing values:\n", df.isnull().sum())

Shape: (400, 17)
Data types:
dev_budget_usd             int64
marketing_budget_usd       int64
dev_time_months            int64
team_size                  int64
metacritic                 int64
user_score                 int64
genre                        str
platforms_count            int64
is_multiplayer             int64
launch_price_usd           int64
median_playtime_hours    float64
release_month              int64
is_franchise               int64
publisher_size               str
reviews_at_launch          int64
launch_discount          float64
first_year_sales_k       float64
dtype: object
Missing values:
 dev_budget_usd           0
marketing_budget_usd     0
dev_time_months          0
team_size                0
metacritic               0
user_score               0
genre                    0
platforms_count          0
is_multiplayer           0
launch_price_usd         0
median_playtime_hours    0
release_month            0
is_franchise             0
publisher_size           0
r

Inspecting the data: column types, missing values, inconsistency.
Column types we got numerical and categorical.
No missing values.
Inconsistencies where checked manually for genre and publisher_size.

In [4]:
import pandasql as ps

# function query(sql) will help us write our SQL queries easily
def query(sql):
    return ps.sqldf(sql, globals())

In [5]:
query("""
SELECT launch_discount AS nr
FROM df
WHERE nr> 0.6 OR nr<0;
""")

,nr


In [6]:
query("""
SELECT metacritic AS rating
FROM df
WHERE rating>100 OR rating<0;
""")

,rating


In [7]:
query("""
SELECT user_score AS score
FROM df
WHERE score> 100 OR score<0;
""")

,score


In [8]:
query("""
SELECT release_month AS release
FROM df
WHERE release>11 OR release<0;
""")

,release


These SQL queries check whether the values in columns are within the given bounds or not. If the values were outside of the bound, the query would return those certain values. Since there are no rows returned, we know that every value is within the required bounds. The bounds were given by the client.

In [9]:
copy_table = df.copy()
rowsToDrop = []
for i, row in copy_table.iterrows():
    if row["dev_budget_usd"] < 3400000:
        rowsToDrop.append(i)

copy_table.drop(rowsToDrop, inplace=True)
print(copy_table)

     dev_budget_usd  marketing_budget_usd  dev_time_months  team_size  \
3           4610783               3308649               51        155   
5           4266193               1274032               57        121   
39          6854661               3014742                6         56   
53          6499290               1850323                7        131   
67          4850363               2396969               31        201   
106        10451118               2717298                8        126   
111         6127719               3383082               56        118   
136         5492024               2783111               23         76   
139         5463647                667811               16        116   
173         9646383               5174662               10        111   
192         4022032                909019               47         80   
393         4053803               1333248               58        512   

     metacritic  user_score      genre  platforms_

We made a code that copies the original dataset and saves in that copy the outliers of the data. We chose 3400000 to be the limit for outliers based on the graph we made (graph given in excel). Since only 12 out of 400 games have a budget over 3.4 million, and there is a clear gap after that.

In [10]:
final_copy_table = df.copy()
rowsToDrop = []
for i, row in final_copy_table.iterrows():
    if row["dev_budget_usd"] > 3400000:
        rowsToDrop.append(i)

final_copy_table.drop(rowsToDrop, inplace=True)
print(final_copy_table)


     dev_budget_usd  marketing_budget_usd  dev_time_months  team_size  \
0           1780590                551738               45        112   
1           2361029                943574                7         23   
2           1976102                829247               37        122   
4           1079161                640524               40         49   
6           1373599                941903                8         96   
..              ...                   ...              ...        ...   
395          408942                229168               51        205   
396          357485                 72742               22         43   
397         1440670                494674               23        122   
398          107511                 45800               59        282   
399          223897                 49966               23        186   

     metacritic  user_score      genre  platforms_count  is_multiplayer  \
0            98         100        rpg          

In the code above we delete the earlier identified outliers. Originally we had 400 rows, now we have 388, meaning we deleted all 12 outliers. However, we did not modify the original data, instead we made a copy specifically for this project.

In [11]:
query("""
SELECT genre, AVG(first_year_sales_k)
FROM final_copy_table
GROUP BY genre;
""")

,genre,AVG(first_year_sales_k)
0,adventure,23.621250
1,puzzle,18.023784
2,rpg,75.070000
3,shooter,70.102750
4,strategy,45.897619


In the query above, we average the first_years_sales_k grouped by genre, to see how genre impacts the sales for the games.

In [12]:
query("""
SELECT genre, platforms_count, AVG(first_year_sales_k)
FROM final_copy_table
GROUP BY genre, platforms_count;
""")

,genre,platforms_count,AVG(first_year_sales_k)
0,adventure,1,25.991600
1,adventure,2,21.169545
2,adventure,3,23.314444
3,adventure,4,23.461250
4,puzzle,1,10.794737
5,puzzle,2,15.490714
6,puzzle,3,23.716087
7,puzzle,4,20.351111
8,rpg,1,79.429565
9,rpg,2,45.968000


In the query above, we average the first_years_sales_k grouped by genre and platform_count, to see how genre and platform_count impacts the sales for the games.

In [13]:
query("""
SELECT 
  CASE 
    WHEN median_playtime_hours < 10 THEN 'Low (<10)'
    WHEN median_playtime_hours BETWEEN 10 AND 40 THEN 'Medium (10-40)'
    WHEN median_playtime_hours BETWEEN 40 AND 50 THEN 'High (40-50)'
    ELSE 'Very High (>50)'
  END AS median_playtime_hours_band,
ROUND(AVG(first_year_sales_k), 2) AS avg_first_year_sales_k
FROM final_copy_table
GROUP BY median_playtime_hours_band
ORDER BY avg_first_year_sales_k DESC;
""")

,median_playtime_hours_band,avg_first_year_sales_k
0,High (40-50),99.30
1,Medium (10-40),45.50
2,Low (<10),44.95
3,Very High (>50),44.12


In the query above, we average the first_years_sales_k grouped by median_playtime_hours_band, to see how that impacts the sales.